# Agregação de Matriz OD por Zoneamento de projeto

Este notebook consolida a rotina completa para agregar a matriz OD de municípios (`vw_mtx_pessoas_auto_calib`) em qualquer zoneamento customizado, tratando dois cenários:

1. **Agregação simples** — quando cada município está 100% contido em uma única zona do novo zoneamento.
2. **Desagregação por setor censitário** — quando um município é cortado por mais de uma zona, as viagens são distribuídas proporcionalmente ao peso dos setores censitários (`cd_sit`) em cada zona. Peso = `1/cd_sit` (setor mais denso pesa mais).

## Estrutura das tabelas envolvidas

| Schema | Tabela | Papel |
|---|---|---|
| `VISUM` | `vw_mtx_pessoas_auto_calib` | Matriz OD por município (colunas `o`, `d`, `viagens`) |
| `VISUM` | `tbl_shp_municipiosibge_2024` | Shapefile municípios IBGE (`cd_mun`, `wkb_geometry`) |
| `VISUM` | `tbl_shp_pequi` (exemplo) | Shapefile do zoneamento novo (`id`, `geom`) |
| `TERRITORIO` | `tbl_zonas_censitarias_2022` | Shapefile de setores censitários (`cd_sit`, geom) |

## Observações importantes

- Os SRIDs diferem entre tabelas: IBGE em **4674** (SIRGAS 2000), zoneamentos em **4326** (WGS 84). Todas as operações espaciais fazem `ST_Transform` quando necessário.
- Os nomes de schema (`VISUM`, `TERRITORIO`) foram criados com letras maiúsculas, então aparecem sempre entre aspas duplas.
- A rotina é **parametrizada**: você passa o nome da tabela de zoneamento e ela faz todo o pipeline.

In [23]:
import os
import signal
import threading
import psycopg2
import pandas as pd
from dotenv import load_dotenv
import warnings
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")

env_path = r"C:\Users\zeno.filho\projetos\python_secrets_lib\.env"
load_dotenv(dotenv_path=env_path)

# Timeouts de segurança (em milissegundos)
STATEMENT_TIMEOUT_MS = 30 * 60 * 1000   # 30 min: mata a query automaticamente
LOCK_TIMEOUT_MS      = 60 * 1000        # 60s: não espera mais que 1 min por lock

# Registro de conexões ativas para podermos cancelá-las
_active_connections = []
_lock = threading.Lock()

def get_db_connection():
    """Abre conexão e configura timeouts de sessão."""
    conn = psycopg2.connect(
        host=os.getenv("DB_HOST"),
        database=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        port=os.getenv("DB_PORT", 5432),
    )
    cur = conn.cursor()
    cur.execute(f"SET statement_timeout = {STATEMENT_TIMEOUT_MS};")
    cur.execute(f"SET lock_timeout = {LOCK_TIMEOUT_MS};")
    cur.close()
    conn.commit()
    return conn


def _register(conn):
    with _lock:
        _active_connections.append(conn)


def _unregister(conn):
    with _lock:
        if conn in _active_connections:
            _active_connections.remove(conn)


def cancel_all():
    """Envia pg_cancel_backend em TODAS as conexões ativas.
    
    Pode ser chamada manualmente a qualquer momento de outra célula
    se você perceber que algo travou.
    """
    with _lock:
        conns = list(_active_connections)
    
    cancelled = 0
    for conn in conns:
        try:
            # Usa a PRÓPRIA conexão travada para enviar o cancel
            # via um canal assíncrono interno do psycopg2
            conn.cancel()
            cancelled += 1
        except Exception as e:
            print(f"Erro ao cancelar: {e}")
    print(f"Cancel enviado para {cancelled} conexão(ões).")


def run_sql(sql, params=None, fetch=False):
    """Executa SQL com cancelamento limpo via Ctrl+C / Interrupt Kernel."""
    conn = get_db_connection()
    _register(conn)
    try:
        if fetch:
            return pd.read_sql(sql, conn, params=params)
        else:
            cur = conn.cursor()
            cur.execute(sql, params)
            conn.commit()
            cur.close()
    except KeyboardInterrupt:
        print("⚠️ Interrupção detectada — enviando cancel ao servidor...")
        try:
            conn.cancel()
        except Exception as e:
            print(f"Erro no cancel: {e}")
        conn.close()
        raise
    except Exception:
        conn.rollback()
        raise
    finally:
        _unregister(conn)
        if not conn.closed:
            conn.close()

def check_locks():
    """Mostra se há alguma sessão segurando locks nas tabelas do zoneamento."""
    sql = '''
    SELECT pid, mode, relation::regclass AS tabela, granted,
           now() - query_start AS tempo,
           LEFT(query, 80) AS query
    FROM pg_locks l
    JOIN pg_stat_activity a USING (pid)
    WHERE relation::regclass::text LIKE '%de_para%'
       OR relation::regclass::text LIKE '%mun_status%'
       OR relation::regclass::text LIKE '%mtx_od%'
    ORDER BY query_start;
    '''
    return run_sql(sql, fetch=True)

# Teste
run_sql("SELECT current_database(), current_user, version();", fetch=True)

,current_database,current_user,version
0,dbs_surod,zeno.andrade,"PostgreSQL 14.6 on x86_64-pc-linux-gnu, compil..."


Checar se tem algo travando o banco

In [ ]:
check_locks()

## EM CASO DE EMERGÊNCIA


In [ ]:
cancel_all()

## 1. Parâmetros do zoneamento

Edite aqui para trocar de zoneamento sem mexer no resto do notebook.

In [5]:
# Schema e tabela do novo zoneamento
ZON_SCHEMA = "VISUM"
ZON_TABELA = "tbl_shp_pequi"       # nome em minúsculo, não precisa de aspas
ZON_COL_ID = "id"                  # coluna identificadora da zona
ZON_COL_GEOM = "geom"              # coluna de geometria
ZON_SRID = 4326                    # SRID da tabela de zoneamento

# Sufixo usado nas tabelas de saída (ex.: de_para_mun_pequi, mtx_od_pequi)
SUFIXO = "pequi"

# Matriz OD de entrada
OD_SCHEMA = "VISUM"
OD_VIEW = "vw_mtx_pessoas_auto_calib"
OD_COL_O = "o"
OD_COL_D = "d"
OD_COL_VIAGENS = "viagens"

# Municípios
MUN_SCHEMA = "VISUM"
MUN_TABELA = "tbl_shp_municipiosibge_2024"
MUN_COL_CD = "cd_mun"
MUN_COL_GEOM = "wkb_geometry"
MUN_SRID = 4674

# Setores censitários
SET_SCHEMA = "TERRITORIO"
SET_TABELA = "tbl_zonas_censitarias_2022"
SET_COL_CDSIT = "cd_sit"
SET_COL_CDMUN = "cd_mun"
SET_COL_GEOM = "wkb_geometry"
SET_SRID = 4674

print(f"Zoneamento atual: {ZON_SCHEMA}.{ZON_TABELA} (sufixo: {SUFIXO})")

Zoneamento atual: VISUM.tbl_shp_pequi (sufixo: pequi)


# 2. Verificações preliminares
Confirmar os SRIDs e a estrutura das tabelas. Ajustr célula anterior se algo não bater.

In [6]:
sql_srids = f'''
SELECT
    (SELECT DISTINCT ST_SRID({MUN_COL_GEOM}) FROM "{MUN_SCHEMA}"."{MUN_TABELA}")    AS srid_municipios,
    (SELECT DISTINCT ST_SRID({ZON_COL_GEOM}) FROM "{ZON_SCHEMA}"."{ZON_TABELA}")    AS srid_zoneamento,
    (SELECT DISTINCT ST_SRID({SET_COL_GEOM}) FROM "{SET_SCHEMA}"."{SET_TABELA}")    AS srid_setores
'''
run_sql(sql_srids, fetch=True)

C:\Users\zeno.filho\AppData\Local\Temp\ipykernel_28312\1751495093.py:75: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,srid_municipios,srid_zoneamento,srid_setores
0,4674,4326,4674


In [7]:
# Verifica tipos das colunas de junção (o/d vs cd_mun) - sensível a mismatch int/varchar
sql_tipos = '''
SELECT table_schema, table_name, column_name, data_type
FROM information_schema.columns
WHERE (table_schema, table_name, column_name) IN (
    (%s, %s, %s),
    (%s, %s, %s),
    (%s, %s, %s)
)
ORDER BY table_schema, table_name, column_name;
'''
params = (
    MUN_SCHEMA, MUN_TABELA, MUN_COL_CD,
    OD_SCHEMA, OD_VIEW, OD_COL_O,
    OD_SCHEMA, OD_VIEW, OD_COL_D,
)
run_sql(sql_tipos, params=params, fetch=True)

C:\Users\zeno.filho\AppData\Local\Temp\ipykernel_28312\1751495093.py:75: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,table_schema,table_name,column_name,data_type
0,VISUM,tbl_shp_municipiosibge_2024,cd_mun,character varying


## 3. De-para município → zona

Gera duas tabelas:

- **`de_para_mun_<sufixo>`** — todos os pares (município, zona) com a área de interseção. Um município que toca N zonas aparece N vezes.
- **`mun_status_<sufixo>`** — marca cada município como `inteiro` (contido em uma única zona) ou `dividido` (cortado por 2+).

Usamos `ST_PointOnSurface` sobre o centroide do município para decidir a zona "principal" quando ele está inteiro. Para os divididos, a desagregação por setor entra em ação na etapa 4.

In [8]:
# Garante índices GIST nas tabelas fonte (idempotente)
sql_gix = f'''
CREATE INDEX IF NOT EXISTS gix_{MUN_TABELA}_geom
  ON "{MUN_SCHEMA}"."{MUN_TABELA}" USING GIST ({MUN_COL_GEOM});
CREATE INDEX IF NOT EXISTS gix_{ZON_TABELA}_geom
  ON "{ZON_SCHEMA}"."{ZON_TABELA}" USING GIST ({ZON_COL_GEOM});
ANALYZE "{MUN_SCHEMA}"."{MUN_TABELA}";
ANALYZE "{ZON_SCHEMA}"."{ZON_TABELA}";
'''
run_sql(sql_gix)

# Corrige geometrias inválidas do zoneamento antes de operar
sql_valid = f'''
UPDATE "{ZON_SCHEMA}"."{ZON_TABELA}"
SET {ZON_COL_GEOM} = ST_MakeValid({ZON_COL_GEOM})
WHERE NOT ST_IsValid({ZON_COL_GEOM});
'''
run_sql(sql_valid)

# De-para e status (otimizado: sem ST_Intersection/ST_Area, com filtro && e _zon_tmp)
sql_depara = f'''
DROP TABLE IF EXISTS "{ZON_SCHEMA}".de_para_mun_{SUFIXO};
DROP TABLE IF EXISTS "{ZON_SCHEMA}".mun_status_{SUFIXO};

-- Zoneamento reprojetado para o SRID dos municípios, uma única vez.
-- Mantendo a geometria dos municípios intocada, o índice GIST deles
-- continua utilizável no JOIN (o que evita varredura total da tabela).
DROP TABLE IF EXISTS _zon_tmp_{SUFIXO};
CREATE TEMP TABLE _zon_tmp_{SUFIXO} AS
SELECT
    {ZON_COL_ID} AS zona_id,
    ST_Transform({ZON_COL_GEOM}, {MUN_SRID}) AS geom
FROM "{ZON_SCHEMA}"."{ZON_TABELA}";

CREATE INDEX ON _zon_tmp_{SUFIXO} USING GIST (geom);
ANALYZE _zon_tmp_{SUFIXO};

-- De-para: pares município x zona que se tocam.
-- Filtro && por bounding box usa índice; ST_Intersects refina.
-- Não calcula área: ponderação posterior é por cd_sit, não por área.
CREATE TABLE "{ZON_SCHEMA}".de_para_mun_{SUFIXO} AS
SELECT
    m.{MUN_COL_CD}::text AS cd_mun,
    z.zona_id
FROM "{MUN_SCHEMA}"."{MUN_TABELA}" m
JOIN _zon_tmp_{SUFIXO} z
  ON m.{MUN_COL_GEOM} && z.geom
 AND ST_Intersects(m.{MUN_COL_GEOM}, z.geom);

CREATE INDEX idx_depara_{SUFIXO}_cdmun ON "{ZON_SCHEMA}".de_para_mun_{SUFIXO}(cd_mun);
CREATE INDEX idx_depara_{SUFIXO}_zona  ON "{ZON_SCHEMA}".de_para_mun_{SUFIXO}(zona_id);

CREATE TABLE "{ZON_SCHEMA}".mun_status_{SUFIXO} AS
SELECT
    cd_mun,
    COUNT(*) AS n_zonas_tocadas,
    CASE WHEN COUNT(*) = 1 THEN 'inteiro' ELSE 'dividido' END AS status
FROM "{ZON_SCHEMA}".de_para_mun_{SUFIXO}
GROUP BY cd_mun;

CREATE INDEX idx_status_{SUFIXO}_cdmun ON "{ZON_SCHEMA}".mun_status_{SUFIXO}(cd_mun);
'''
run_sql(sql_depara)
print("De-para e status criados.")

De-para e status criados.


In [9]:
# Resumo: quantos municípios inteiros x divididos
sql_resumo = f'''
SELECT status, COUNT(*) AS n_municipios
FROM "{ZON_SCHEMA}".mun_status_{SUFIXO}
GROUP BY status
ORDER BY status;
'''
run_sql(sql_resumo, fetch=True)

C:\Users\zeno.filho\AppData\Local\Temp\ipykernel_28312\1751495093.py:75: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,status,n_municipios
0,dividido,1943
1,inteiro,3629


## 4. Pesos por setor censitário

Para cada município **dividido**, calculamos o peso de cada zona nova como:

$$
\text{peso}_{(\text{mun}, \text{zona})} = \sum_{\text{setor} \in \text{mun} \cap \text{zona}} \frac{1}{\text{cd\_sit}_{\text{setor}}}
$$
`cd_sit` classifica a zona conforme sua densidade, variando de 1 (mais densa) a 9 (menos densa). 

A fração das viagens do município que vai para cada zona é esse peso normalizado pelo peso total do município.

**Como decidimos em qual zona um setor está?** Usamos o centroide (`ST_PointOnSurface`) do setor contra o polígono da zona — setor inteiro cai em uma zona só, coerente com a premissa que é utilizada para municípios inteiros.

**Atenção:** essa etapa só considera setores dos municípios divididos. Os inteiros não precisam de pesagem.

In [10]:
sql_pesos = f'''
DROP TABLE IF EXISTS "{ZON_SCHEMA}".pesos_mun_zona_{SUFIXO};

CREATE TABLE "{ZON_SCHEMA}".pesos_mun_zona_{SUFIXO} AS
WITH setores_muni_dividido AS (
    -- Só olha setores dos municípios que precisam de desagregação
    SELECT s.*
    FROM "{SET_SCHEMA}"."{SET_TABELA}" s
    JOIN "{ZON_SCHEMA}".mun_status_{SUFIXO} ms
      ON s.{SET_COL_CDMUN}::text = ms.cd_mun
    WHERE ms.status = 'dividido'
),
setor_para_zona AS (
    -- Cada setor é atribuído à zona nova que contém seu centroide
    SELECT
        s.{SET_COL_CDMUN}::text AS cd_mun,
        z.{ZON_COL_ID}          AS zona_id,
        s.{SET_COL_CDSIT}       AS cd_sit,
        (1.0 / NULLIF(s.{SET_COL_CDSIT}::numeric, 0)) AS peso_setor
    FROM setores_muni_dividido s
    JOIN "{ZON_SCHEMA}"."{ZON_TABELA}" z
      ON ST_Contains(
           z.{ZON_COL_GEOM},
           ST_Transform(ST_PointOnSurface(s.{SET_COL_GEOM}), {ZON_SRID})
         )
)
SELECT
    cd_mun,
    zona_id,
    SUM(peso_setor) AS peso_zona,
    COUNT(*)        AS n_setores
FROM setor_para_zona
GROUP BY cd_mun, zona_id;

-- Fração normalizada por município
ALTER TABLE "{ZON_SCHEMA}".pesos_mun_zona_{SUFIXO}
  ADD COLUMN fracao numeric;

UPDATE "{ZON_SCHEMA}".pesos_mun_zona_{SUFIXO} p
SET fracao = p.peso_zona / t.peso_total
FROM (
    SELECT cd_mun, SUM(peso_zona) AS peso_total
    FROM "{ZON_SCHEMA}".pesos_mun_zona_{SUFIXO}
    GROUP BY cd_mun
) t
WHERE p.cd_mun = t.cd_mun;

CREATE INDEX idx_pesos_{SUFIXO}_cdmun ON "{ZON_SCHEMA}".pesos_mun_zona_{SUFIXO}(cd_mun);
CREATE INDEX idx_pesos_{SUFIXO}_zona  ON "{ZON_SCHEMA}".pesos_mun_zona_{SUFIXO}(zona_id);
'''
run_sql(sql_pesos)
print("Pesos município x zona calculados.")

Pesos município x zona calculados.


In [11]:
# Sanidade: a soma das frações por município deve ser 1.0 (ou muito próxima)
sql_check_fracao = f'''
SELECT
    cd_mun,
    ROUND(SUM(fracao)::numeric, 6) AS soma_fracao,
    COUNT(*) AS n_zonas
FROM "{ZON_SCHEMA}".pesos_mun_zona_{SUFIXO}
GROUP BY cd_mun
HAVING ROUND(SUM(fracao)::numeric, 6) <> 1.0
ORDER BY cd_mun
LIMIT 20;
'''
df_check = run_sql(sql_check_fracao, fetch=True)
if df_check.empty:
    print("✅ Todas as frações somam 1 por município.")
else:
    print("⚠️ Municípios com soma de fração diferente de 1 (pode ser arredondamento):")
    print(df_check)

✅ Todas as frações somam 1 por município.


C:\Users\zeno.filho\AppData\Local\Temp\ipykernel_28312\1751495093.py:75: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


## 5. Tabela unificada de rateio município → zona

Agora unimos os dois casos em uma única tabela:

- Município **inteiro** → fração = 1.0 para a zona que o contém.
- Município **dividido** → uma linha por zona com a fração calculada na etapa anterior.

Essa tabela é o que vamos usar pra agregar a matriz OD.

In [12]:
sql_rateio = f'''
DROP TABLE IF EXISTS "{ZON_SCHEMA}".rateio_mun_zona_{SUFIXO};

CREATE TABLE "{ZON_SCHEMA}".rateio_mun_zona_{SUFIXO} AS
-- Caso 1: municípios inteiros (fração = 1)
SELECT
    d.cd_mun,
    d.zona_id,
    1.0::numeric AS fracao
FROM "{ZON_SCHEMA}".de_para_mun_{SUFIXO} d
JOIN "{ZON_SCHEMA}".mun_status_{SUFIXO} ms
  ON d.cd_mun = ms.cd_mun
WHERE ms.status = 'inteiro'

UNION ALL

-- Caso 2: municípios divididos (fração calculada pelos setores)
SELECT
    p.cd_mun,
    p.zona_id,
    p.fracao
FROM "{ZON_SCHEMA}".pesos_mun_zona_{SUFIXO} p;

CREATE INDEX idx_rateio_{SUFIXO}_cdmun ON "{ZON_SCHEMA}".rateio_mun_zona_{SUFIXO}(cd_mun);
CREATE INDEX idx_rateio_{SUFIXO}_zona  ON "{ZON_SCHEMA}".rateio_mun_zona_{SUFIXO}(zona_id);
'''
run_sql(sql_rateio)
print("Tabela de rateio unificada criada.")

Tabela de rateio unificada criada.


In [13]:
# Verifica cobertura: todo município com OD precisa estar no rateio
sql_cobertura = f'''
WITH mun_na_matriz AS (
    SELECT {OD_COL_O}::text AS cd_mun FROM "{OD_SCHEMA}"."{OD_VIEW}"
    UNION
    SELECT {OD_COL_D}::text AS cd_mun FROM "{OD_SCHEMA}"."{OD_VIEW}"
)
SELECT COUNT(*) AS municipios_sem_rateio
FROM mun_na_matriz m
LEFT JOIN "{ZON_SCHEMA}".rateio_mun_zona_{SUFIXO} r ON m.cd_mun = r.cd_mun
WHERE r.cd_mun IS NULL;
'''
run_sql(sql_cobertura, fetch=True)

C:\Users\zeno.filho\AppData\Local\Temp\ipykernel_28312\1751495093.py:75: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,municipios_sem_rateio
0,1


## 6. Agregação da matriz OD

Para cada par origem-destino da matriz original, expandimos usando o rateio dos dois lados:

$$
\text{viagens}_{(Z_o, Z_d)} = \sum_{(m_o, m_d)} \text{viagens}_{(m_o, m_d)} \cdot f_{(m_o, Z_o)} \cdot f_{(m_d, Z_d)}
$$

Onde $f$ é a fração do rateio. Para municípios inteiros, $f = 1$, então o comportamento é idêntico ao caso simples.

In [14]:
sql_agregacao = f'''
DROP TABLE IF EXISTS "{ZON_SCHEMA}".mtx_od_{SUFIXO};

CREATE TABLE "{ZON_SCHEMA}".mtx_od_{SUFIXO} AS
SELECT
    ro.zona_id AS o,
    rd.zona_id AS d,
    SUM(od.{OD_COL_VIAGENS} * ro.fracao * rd.fracao) AS viagens
FROM "{OD_SCHEMA}"."{OD_VIEW}" od
JOIN "{ZON_SCHEMA}".rateio_mun_zona_{SUFIXO} ro ON od.{OD_COL_O}::text = ro.cd_mun
JOIN "{ZON_SCHEMA}".rateio_mun_zona_{SUFIXO} rd ON od.{OD_COL_D}::text = rd.cd_mun
GROUP BY ro.zona_id, rd.zona_id
ORDER BY ro.zona_id, rd.zona_id;

CREATE INDEX idx_mtx_{SUFIXO}_o ON "{ZON_SCHEMA}".mtx_od_{SUFIXO}(o);
CREATE INDEX idx_mtx_{SUFIXO}_d ON "{ZON_SCHEMA}".mtx_od_{SUFIXO}(d);
'''
run_sql(sql_agregacao)
print("Matriz OD agregada criada.")

Matriz OD agregada criada.


## 7. Validações

A soma de viagens original tem que ser **praticamente igual** à soma agregada (pode haver diferença mínima de ponto flutuante). Se a diferença for grande, provavelmente há municípios na matriz OD que não estão cobertos pelo zoneamento.

In [15]:
sql_totais = f'''
SELECT
    (SELECT SUM({OD_COL_VIAGENS}) FROM "{OD_SCHEMA}"."{OD_VIEW}") AS total_original,
    (SELECT SUM(viagens)          FROM "{ZON_SCHEMA}".mtx_od_{SUFIXO}) AS total_agregado
'''
df_totais = run_sql(sql_totais, fetch=True)
df_totais["diff_pct"] = (
    (df_totais["total_agregado"] - df_totais["total_original"])
    / df_totais["total_original"] * 100
)
df_totais

C:\Users\zeno.filho\AppData\Local\Temp\ipykernel_28312\1751495093.py:75: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,total_original,total_agregado,diff_pct
0,2.239158e+09,2.254412e+09,0.681225


In [16]:
# Contagem de pares OD antes e depois
sql_pares = f'''
SELECT
    (SELECT COUNT(*) FROM "{OD_SCHEMA}"."{OD_VIEW}")          AS pares_original,
    (SELECT COUNT(*) FROM "{ZON_SCHEMA}".mtx_od_{SUFIXO})     AS pares_agregado
'''
run_sql(sql_pares, fetch=True)

C:\Users\zeno.filho\AppData\Local\Temp\ipykernel_28312\1751495093.py:75: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,pares_original,pares_agregado
0,4062465,42884


In [17]:
# Top 10 pares do resultado, como preview
sql_preview = f'''
SELECT o, d, viagens
FROM "{ZON_SCHEMA}".mtx_od_{SUFIXO}
ORDER BY viagens DESC
LIMIT 10;
'''
run_sql(sql_preview, fetch=True)

C:\Users\zeno.filho\AppData\Local\Temp\ipykernel_28312\1751495093.py:75: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,o,d,viagens
0,218.0,218.0,3.735605e+08
1,224.0,224.0,2.926307e+08
2,232.0,232.0,2.894320e+08
3,206.0,206.0,1.575000e+08
4,228.0,228.0,4.013994e+07
5,216.0,216.0,2.881818e+07
6,199.0,199.0,2.815101e+07
7,230.0,230.0,2.494609e+07
8,216.0,218.0,2.479787e+07
9,218.0,216.0,2.479787e+07


## 8. Função reutilizável

A função abaixo consolida todas as etapas. Útil quando você precisa testar vários zoneamentos em sequência.

In [18]:
# Referências fixas — não altere
_MUN_SCHEMA   = "VISUM"
_MUN_TABELA   = "tbl_shp_municipiosibge_2024"
_MUN_COL_CD   = "cd_mun"
_MUN_COL_GEOM = "wkb_geometry"
_MUN_SRID     = 4674

_SET_SCHEMA    = "TERRITORIO"
_SET_TABELA    = "tbl_zonas_censitarias_2022"
_SET_COL_CDSIT = "cd_sit"
_SET_COL_CDMUN = "cd_mun"
_SET_COL_GEOM  = "wkb_geometry"
_SET_SRID      = 4674

In [19]:
def agregar_od_por_zoneamento(
    # ── Matriz OD de origem ──────────────────────────────────────
    od_schema,       # schema da tabela/view de OD         ex: "VISUM"
    od_view,         # nome da tabela/view de OD           ex: "vw_mtx_pessoas_auto_calib"
    od_col_o,        # coluna de origem                    ex: "o"
    od_col_d,        # coluna de destino                   ex: "d"
    od_col_viagens,  # coluna de viagens                   ex: "viagens"
    # ── Zoneamento de destino ─────────────────────────────────────
    zon_schema,      # schema da tabela de zoneamento      ex: "VISUM"
    zon_tabela,      # nome da tabela de zoneamento        ex: "tbl_shp_pequi"
    zon_col_id,      # coluna identificadora da zona       ex: "id"
    zon_col_geom,    # coluna de geometria                 ex: "geom"
    zon_srid,        # SRID do zoneamento                  ex: 4326
    sufixo,          # sufixo das tabelas de saída         ex: "pequi"
    validar_geometrias=True,
):
    """Pipeline completa de agregação/desagregação da matriz OD para um novo zoneamento.

    Incorpora as otimizações:
      - Garante índices GIST em todas as tabelas fonte (IF NOT EXISTS).
      - Corrige geometrias inválidas do zoneamento com ST_MakeValid.
      - Reprojeta o zoneamento para o SRID dos municípios uma única vez,
        em tabela temporária indexada, para preservar o uso do índice GIST
        na tabela grande (municípios).
      - Filtra pares candidatos com && (bounding box) antes de ST_Intersects.
      - Não calcula ST_Intersection / ST_Area no de-para (ponderação é por cd_sit).

    Tabelas criadas em zon_schema:
      de_para_mun_<sufixo>, mun_status_<sufixo>, pesos_mun_zona_<sufixo>,
      rateio_mun_zona_<sufixo>, mtx_od_<sufixo>

    Retorna dict com total_original, total_agregado, diff_pct.
    """
    ctx = dict(
        ZS=zon_schema, ZT=zon_tabela, ZID=zon_col_id, ZG=zon_col_geom, ZSRID=zon_srid,
        SF=sufixo,
        OS=od_schema, OV=od_view, OO=od_col_o, OD=od_col_d, OV_=od_col_viagens,
        MS=_MUN_SCHEMA, MT=_MUN_TABELA, MC=_MUN_COL_CD, MG=_MUN_COL_GEOM, MSRID=_MUN_SRID,
        SS=_SET_SCHEMA, ST=_SET_TABELA, SC=_SET_COL_CDSIT, SM=_SET_COL_CDMUN,
        SG=_SET_COL_GEOM, SSRID=_SET_SRID,
    )

    # 0. Índices GIST nas tabelas fonte (idempotente)
    sql_idx = '''
    CREATE INDEX IF NOT EXISTS gix_{MT}_geom ON "{MS}"."{MT}" USING GIST ({MG});
    CREATE INDEX IF NOT EXISTS gix_{ZT}_geom ON "{ZS}"."{ZT}" USING GIST ({ZG});
    CREATE INDEX IF NOT EXISTS gix_{ST}_geom ON "{SS}"."{ST}" USING GIST ({SG});
    ANALYZE "{MS}"."{MT}";
    ANALYZE "{ZS}"."{ZT}";
    ANALYZE "{SS}"."{ST}";
    '''.format(**ctx)
    run_sql(sql_idx)

    # 1. Validação opcional de geometrias do zoneamento
    if validar_geometrias:
        sql_valid = '''
        UPDATE "{ZS}"."{ZT}"
        SET {ZG} = ST_MakeValid({ZG})
        WHERE NOT ST_IsValid({ZG});
        '''.format(**ctx)
        run_sql(sql_valid)

    # 2. De-para + status (via tabela temporária do zoneamento reprojetado)
    sql_depara = '''
    DROP TABLE IF EXISTS "{ZS}".de_para_mun_{SF};
    DROP TABLE IF EXISTS "{ZS}".mun_status_{SF};
    DROP TABLE IF EXISTS "{ZS}".pesos_mun_zona_{SF};
    DROP TABLE IF EXISTS "{ZS}".rateio_mun_zona_{SF};
    DROP TABLE IF EXISTS "{ZS}".mtx_od_{SF};

    DROP TABLE IF EXISTS _zon_tmp_{SF};
    CREATE TEMP TABLE _zon_tmp_{SF} AS
    SELECT {ZID} AS zona_id, ST_Transform({ZG}, {MSRID}) AS geom
    FROM "{ZS}"."{ZT}";
    CREATE INDEX ON _zon_tmp_{SF} USING GIST (geom);
    ANALYZE _zon_tmp_{SF};

    CREATE TABLE "{ZS}".de_para_mun_{SF} AS
    SELECT m.{MC}::text AS cd_mun, z.zona_id
    FROM "{MS}"."{MT}" m
    JOIN _zon_tmp_{SF} z
      ON m.{MG} && z.geom
     AND ST_Intersects(m.{MG}, z.geom);

    CREATE INDEX idx_depara_{SF}_cdmun ON "{ZS}".de_para_mun_{SF}(cd_mun);
    CREATE INDEX idx_depara_{SF}_zona  ON "{ZS}".de_para_mun_{SF}(zona_id);

    CREATE TABLE "{ZS}".mun_status_{SF} AS
    SELECT cd_mun, COUNT(*) AS n_zonas_tocadas,
           CASE WHEN COUNT(*) = 1 THEN 'inteiro' ELSE 'dividido' END AS status
    FROM "{ZS}".de_para_mun_{SF}
    GROUP BY cd_mun;

    CREATE INDEX idx_status_{SF}_cdmun ON "{ZS}".mun_status_{SF}(cd_mun);
    '''.format(**ctx)
    run_sql(sql_depara)

    # 3. Pesos por cd_sit (para municípios divididos)
    sql_pesos = '''
    DROP TABLE IF EXISTS _zon_tmp_{SF};
    CREATE TEMP TABLE _zon_tmp_{SF} AS
    SELECT {ZID} AS zona_id, ST_Transform({ZG}, {MSRID}) AS geom
    FROM "{ZS}"."{ZT}";
    CREATE INDEX ON _zon_tmp_{SF} USING GIST (geom);
    ANALYZE _zon_tmp_{SF};

    CREATE TABLE "{ZS}".pesos_mun_zona_{SF} AS
    WITH setores_div AS (
        SELECT s.*
        FROM "{SS}"."{ST}" s
        JOIN "{ZS}".mun_status_{SF} ms ON s.{SM}::text = ms.cd_mun
        WHERE ms.status = 'dividido'
    ),
    setor_ponto AS (
        SELECT
            s.{SM}::text AS cd_mun,
            s.{SC}       AS cd_sit,
            CASE WHEN ST_SRID(s.{SG}) = {MSRID}
                 THEN ST_PointOnSurface(s.{SG})
                 ELSE ST_Transform(ST_PointOnSurface(s.{SG}), {MSRID})
            END AS pt
        FROM setores_div s
    ),
    setor_em_zona AS (
        SELECT sp.cd_mun, z.zona_id, sp.cd_sit,
               (1.0 / NULLIF(sp.cd_sit::numeric, 0)) AS peso_setor
        FROM setor_ponto sp
        JOIN _zon_tmp_{SF} z ON ST_Contains(z.geom, sp.pt)
    )
    SELECT cd_mun, zona_id,
           SUM(peso_setor) AS peso_zona,
           COUNT(*)        AS n_setores
    FROM setor_em_zona
    GROUP BY cd_mun, zona_id;

    ALTER TABLE "{ZS}".pesos_mun_zona_{SF} ADD COLUMN fracao numeric;
    UPDATE "{ZS}".pesos_mun_zona_{SF} p
    SET fracao = p.peso_zona / t.peso_total
    FROM (SELECT cd_mun, SUM(peso_zona) AS peso_total
          FROM "{ZS}".pesos_mun_zona_{SF} GROUP BY cd_mun) t
    WHERE p.cd_mun = t.cd_mun;

    CREATE INDEX idx_pesos_{SF}_cdmun ON "{ZS}".pesos_mun_zona_{SF}(cd_mun);
    CREATE INDEX idx_pesos_{SF}_zona  ON "{ZS}".pesos_mun_zona_{SF}(zona_id);
    '''.format(**ctx)
    run_sql(sql_pesos)

    # 4. Rateio unificado (inteiros + divididos)
    sql_rateio = '''
    CREATE TABLE "{ZS}".rateio_mun_zona_{SF} AS
    SELECT d.cd_mun, d.zona_id, 1.0::numeric AS fracao
    FROM "{ZS}".de_para_mun_{SF} d
    JOIN "{ZS}".mun_status_{SF} ms ON d.cd_mun = ms.cd_mun
    WHERE ms.status = 'inteiro'
    UNION ALL
    SELECT cd_mun, zona_id, fracao FROM "{ZS}".pesos_mun_zona_{SF};

    CREATE INDEX idx_rateio_{SF}_cdmun ON "{ZS}".rateio_mun_zona_{SF}(cd_mun);
    CREATE INDEX idx_rateio_{SF}_zona  ON "{ZS}".rateio_mun_zona_{SF}(zona_id);
    '''.format(**ctx)
    run_sql(sql_rateio)

    # 5. Agregação da matriz OD
    sql_mtx = '''
    CREATE TABLE "{ZS}".mtx_od_{SF} AS
    SELECT ro.zona_id AS o, rd.zona_id AS d,
           SUM(od.{OV_} * ro.fracao * rd.fracao) AS viagens
    FROM "{OS}"."{OV}" od
    JOIN "{ZS}".rateio_mun_zona_{SF} ro ON od.{OO}::text = ro.cd_mun
    JOIN "{ZS}".rateio_mun_zona_{SF} rd ON od.{OD}::text = rd.cd_mun
    GROUP BY ro.zona_id, rd.zona_id;

    CREATE INDEX idx_mtx_{SF}_o ON "{ZS}".mtx_od_{SF}(o);
    CREATE INDEX idx_mtx_{SF}_d ON "{ZS}".mtx_od_{SF}(d);
    '''.format(**ctx)
    run_sql(sql_mtx)

    # 6. Validação
    sql_val = '''
    SELECT
        (SELECT SUM({OV_}) FROM "{OS}"."{OV}")        AS total_original,
        (SELECT SUM(viagens) FROM "{ZS}".mtx_od_{SF}) AS total_agregado
    '''.format(**ctx)
    df = run_sql(sql_val, fetch=True)
    tot_orig = float(df.iloc[0]["total_original"])
    tot_agg  = float(df.iloc[0]["total_agregado"])
    diff_pct = (tot_agg - tot_orig) / tot_orig * 100 if tot_orig else None

    return {
        "sufixo": sufixo,
        "tabela_resultado": f'"{zon_schema}".mtx_od_{sufixo}',
        "total_original": tot_orig,
        "total_agregado": tot_agg,
        "diff_pct": diff_pct,
    }

In [20]:
resultado = agregar_od_por_zoneamento(
    # Matriz OD de origem
    od_schema      = "VISUM",
    od_view        = "vw_mtx_pessoas_auto_calib",
    od_col_o       = "o",
    od_col_d       = "d",
    od_col_viagens = "viagens",
    # Zoneamento de destino
    zon_schema  = "VISUM",
    zon_tabela  = "tbl_shp_pequi",
    zon_col_id  = "id",
    zon_col_geom= "geom",
    zon_srid    = 4326,
    sufixo      = "pequi",
)
print(resultado)

C:\Users\zeno.filho\AppData\Local\Temp\ipykernel_28312\1751495093.py:75: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


{'sufixo': 'pequi', 'tabela_resultado': '"VISUM".mtx_od_pequi', 'total_original': 2239158500.0, 'total_agregado': 2254412217.936544, 'diff_pct': 0.6812254664662615}


In [24]:
# ------------------------------------------------------------------
# Relatório 1: volume de viagens "fora" do zoneamento
# ------------------------------------------------------------------
sql_fora = f'''
WITH fora AS (
    SELECT od.*
    FROM "{OD_SCHEMA}"."{OD_VIEW}" od
    LEFT JOIN "{ZON_SCHEMA}".rateio_mun_zona_{SUFIXO} ro ON od.{OD_COL_O}::text = ro.cd_mun
    LEFT JOIN "{ZON_SCHEMA}".rateio_mun_zona_{SUFIXO} rd ON od.{OD_COL_D}::text = rd.cd_mun
    WHERE ro.cd_mun IS NULL OR rd.cd_mun IS NULL
),
totais AS (
    SELECT SUM({OD_COL_VIAGENS}) AS total FROM "{OD_SCHEMA}"."{OD_VIEW}"
)
SELECT
    (SELECT COUNT(*)              FROM fora)   AS pares_fora,
    (SELECT SUM({OD_COL_VIAGENS}) FROM fora)   AS viagens_fora,
    (SELECT total FROM totais)                 AS total_viagens,
    ROUND(
        (((SELECT SUM({OD_COL_VIAGENS}) FROM fora)
          / NULLIF((SELECT total FROM totais), 0)) * 100)::numeric,
        4
    ) AS pct_fora
'''
df_fora = run_sql(sql_fora, fetch=True)
print("▸ Volume fora do zoneamento:")
print(df_fora.to_string(index=False))
print()

# ------------------------------------------------------------------
# Relatório 2: municípios da matriz OD que não estão no rateio
# ------------------------------------------------------------------
sql_mun_fora = f'''
WITH mun_na_matriz AS (
    SELECT {OD_COL_O}::text AS cd_mun, SUM({OD_COL_VIAGENS}) AS v_origem, 0::numeric AS v_destino
    FROM "{OD_SCHEMA}"."{OD_VIEW}" GROUP BY {OD_COL_O}
    UNION ALL
    SELECT {OD_COL_D}::text, 0::numeric, SUM({OD_COL_VIAGENS})
    FROM "{OD_SCHEMA}"."{OD_VIEW}" GROUP BY {OD_COL_D}
),
consolidado AS (
    SELECT cd_mun, SUM(v_origem) AS viagens_origem, SUM(v_destino) AS viagens_destino
    FROM mun_na_matriz GROUP BY cd_mun
)
SELECT
    c.cd_mun,
    mun.nm_mun,
    mun.sigla_uf,
    c.viagens_origem,
    c.viagens_destino,
    (c.viagens_origem + c.viagens_destino) AS viagens_total
FROM consolidado c
LEFT JOIN "{ZON_SCHEMA}".rateio_mun_zona_{SUFIXO} r ON c.cd_mun = r.cd_mun
LEFT JOIN "{MUN_SCHEMA}"."{MUN_TABELA}" mun ON c.cd_mun = mun.{MUN_COL_CD}::text
WHERE r.cd_mun IS NULL
ORDER BY viagens_total DESC;
'''
df_mun_fora = run_sql(sql_mun_fora, fetch=True)
print(f"▸ Municípios da matriz OD sem rateio ({len(df_mun_fora)} municípios):")
if df_mun_fora.empty:
    print("  ✅ Nenhum — todos os municípios da matriz estão cobertos pelo zoneamento.")
else:
    print(df_mun_fora.head(20).to_string(index=False))
    if len(df_mun_fora) > 20:
        print(f"  ... e mais {len(df_mun_fora) - 20} municípios.")
print()

# ------------------------------------------------------------------
# Relatório 3: cobertura geral do shapefile IBGE
# ------------------------------------------------------------------
sql_cobertura_geral = f'''
SELECT
    CASE WHEN r.cd_mun IS NULL THEN 'fora_do_zoneamento' ELSE 'dentro_do_zoneamento' END AS cobertura,
    COUNT(DISTINCT mun.{MUN_COL_CD}) AS n_municipios
FROM "{MUN_SCHEMA}"."{MUN_TABELA}" mun
LEFT JOIN "{ZON_SCHEMA}".rateio_mun_zona_{SUFIXO} r ON mun.{MUN_COL_CD}::text = r.cd_mun
GROUP BY cobertura
ORDER BY cobertura;
'''
df_cob = run_sql(sql_cobertura_geral, fetch=True)
print("▸ Cobertura geral do IBGE pelo zoneamento:")
print(df_cob.to_string(index=False))
print()

# ------------------------------------------------------------------
# Diagnóstico final: a diferença observada é explicada?
# ------------------------------------------------------------------
pct_fora = float(df_fora.iloc[0]["pct_fora"] or 0)
print(f"▸ Interpretação:")
if pct_fora > 0:
    print(f"  A diferença no total agregado é ~{pct_fora:.4f}% do volume original.")
    print(f"  Se esse número bate com a 'diff_pct' da etapa 7, a perda é inteiramente")
    print(f"  explicada por municípios fora da área coberta pelo zoneamento.")
else:
    print("  Nenhuma viagem da matriz OD está fora do zoneamento — cobertura total.")

▸ Volume fora do zoneamento:
 pares_fora  viagens_fora  total_viagens  pct_fora
       2896     2505263.2   2239158500.0    0.1119

▸ Municípios da matriz OD sem rateio (1 municípios):
 cd_mun   nm_mun sigla_uf  viagens_origem  viagens_destino  viagens_total
3520400 Ilhabela       SP       1248814.0        1249561.5      2498375.5

▸ Cobertura geral do IBGE pelo zoneamento:
           cobertura  n_municipios
dentro_do_zoneamento          5572
  fora_do_zoneamento             1

▸ Interpretação:
  A diferença no total agregado é ~0.1119% do volume original.
  Se esse número bate com a 'diff_pct' da etapa 7, a perda é inteiramente
  explicada por municípios fora da área coberta pelo zoneamento.


## Notas finais

- **Performance**: se a matriz for muito grande (milhões de pares), `REFRESH MATERIALIZED VIEW "VISUM".vw_mtx_pessoas_auto_calib` antes da agregação pra garantir dados frescos, e considere `VACUUM ANALYZE` nas tabelas de rateio criadas.
- **SRIDs**: toda operação espacial converte para o SRID do zoneamento (`ZON_SRID`). Se um dia o zoneamento vier em SIRGAS 2000 direto, basta mudar o parâmetro.
- **Peso alternativo**: se mais tarde você quiser experimentar outra função de peso (ex.: população do setor, domicílios), é só trocar a expressão `1.0 / cd_sit` na CTE `sp` da função.
- **Edge case**: se um setor cair fora de qualquer polígono do zoneamento (o que teoricamente não deveria ocorrer), ele some do cálculo de peso. Na prática, para municípios divididos, vale rodar uma validação cruzada contando setores esperados vs pesados.